In [ ]:
# Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved.
#
# SPDX-License-Identifier: BSD-3-Clause
#
# Portions of this file consist of AI-generated content. AI-assisted
# content has been reviewed and validated by the authors.

# Language on the wire: Gemma's command as a ROS 2 node

Gemma 4 hears one spoken sentence and answers with one command and its
object. This notebook is the ROS 2 boundary after that:
the node publishes the command as `{"command", "object"}`, and the rest of
the robot never links against the model.

![Alt Text](../../animations/Intent_Node_Animation.gif)

The live path speaks. This notebook publishes a command as text on
`/vla/transcript` so you can drive the node without a microphone. A
terminal, a test, or the console can publish the same way:
`ros2 topic pub /vla/transcript ...` and the robot takes the command.
The node does not know whether the text came from Gemma or from you.

Two design rules carry over and are the heart of your TODOs:

1. **The stop fast path.** `is_stop()` keyword-matches the RAW transcript
   and publishes immediately. The safety stop never waits for a language
   model.
2. **The noise gate.** Upstream text sources deliver junk sometimes:
   empty strings, boilerplate like "Thank you.", fragments. With a grammar
   that *guarantees* a valid command out, a junk transcript would become a
   real robot behavior. `is_noise()` drops it before the LLM.

Also note the QoS flip: transcripts and intents use **RELIABLE** delivery
(the rclpy default). A lost camera frame is free; a lost "stop" is not.

**Matching TODOs:** `2.10`-`2.11` in `project/ros/intent_node.py`.


In [ ]:
# setup: the pipeline (vla_pipeline) and the workshop's helpers (vvla_workshop)
# are installed in this kernel's environment; your TODOs are in workshop/project/.
import threading, time
import json

from vvla_workshop.quiet import hush_benign_warnings

hush_benign_warnings()  # silence known-benign third-party log noise

from vla_pipeline.utils.config import load_config, resolve
import ipywidgets as widgets
from IPython.display import display
from jupyter_ui_poll import ui_events
import requests
from vla_pipeline.llm.intents import INTENT_SYSTEM_PROMPT, Intent, ParsedCommand, is_noise, is_stop, keyword_parse

cfg = load_config()

In [ ]:
# ROS 2 check: this notebook runs on real ROS 2 (rclpy) only.
try:
    import rclpy

    HAVE_ROS = True
except ImportError:
    HAVE_ROS = False

ROS_HELP = """rclpy is not importable in this kernel, so the ROS 2 cells cannot run."""


def require_ros():
    if not HAVE_ROS:
        raise RuntimeError(ROS_HELP)


if HAVE_ROS:
    if not rclpy.ok():
        rclpy.init()
    print("real ROS 2 (rclpy) ready")
else:
    print(ROS_HELP)

> **About background nodes.** The demo nodes this notebook creates are
> self-contained: they publish their own data, so the lesson works with
> nothing else running. To also run the *real* device nodes (the SO-101
> arm, the cameras, the intent node), start them yourself in a terminal;
> the "On the rig" section below has the exact commands. Every subscriber
> simply reads whatever is on the topic, so both can coexist.

In [ ]:
require_ros()

# A tiny helper to spin nodes on background executors, so the notebook
# stays interactive while nodes tick. (This mirrors how the pipeline's
# orchestrator embeds its client nodes.)


class Runner:
    def __init__(self, node):
        from rclpy.executors import SingleThreadedExecutor

        self.node, self.executor = node, SingleThreadedExecutor()
        self.executor.add_node(node)
        threading.Thread(target=self.executor.spin, daemon=True).start()

    def stop(self):
        self.executor.shutdown()
        try:
            self.node.destroy_node()
        except Exception:
            pass

## 1. A minimal intent node

The parser is **injected** (`str -> ParsedCommand`). The pipeline builds
it as Gemma 4 E4B on the iGPU (`LlamaIntentParser` on the `gemma4-e4b-it`
section, GBNF grammar attached) whenever llama-server is reachable, with
the keyword stand-in only as a fallback so the lesson runs on any machine.
The difference matters: free language like *"bust a move"* only routes to
`dance` through the real brain; the keyword stand-in knows only the
literal verbs. The node doesn't know or care which it got - that seam is
what the harness exploits to test your wiring without a GPU. We wrap the
parser to *count calls*, because the whole point of section 3 is proving
certain messages never reach it.


In [ ]:
require_ros()

from rclpy.node import Node
from std_msgs.msg import String

# Gemma 4 when the server is reachable, else the keyword stand-in.
llama_parser = None


def make_parser():
    global llama_parser
    try:
        from vla_pipeline.llm.llama_intent import LlamaIntentParser

        llama_parser = LlamaIntentParser(cfg)  # gemma4-e4b-it section
        print("parser: Gemma 4 E4B on the iGPU (grammar-constrained)")
        return llama_parser.parse
    except Exception as e:
        print(f"parser: Gemma is not ready ({e}) - keyword stand-in")
        return keyword_parse


_parse = make_parser()

parser_calls = []


def demo_parser(text):
    parser_calls.append(text)  # count every LLM "round trip"
    return _parse(text)


class DemoIntentNode(Node):
    def __init__(self, parser):
        super().__init__("demo_intent_node")
        self._parse = parser
        # RELIABLE (default QoS, depth 10): intents are commands, not sensor data.
        self.create_subscription(String, "/vla/transcript", self.on_transcript, 10)
        self.pub = self.create_publisher(String, "/vla/intent", 10)

    def on_transcript(self, msg):
        text = msg.data or ""
        if is_stop(text):  # 1) safety first, LLM never
            self.emit(ParsedCommand(Intent.STOP, transcript=text))
            return
        if is_noise(text):  # 2) junk gate
            return
        self.emit(self._parse(text))  # 3) the brain

    def emit(self, cmd):
        self.pub.publish(
            String(
                data=json.dumps(
                    {"command": cmd.intent.value, "object": cmd.object_name or ""}
                )
            )
        )


intent_node = DemoIntentNode(demo_parser)
node_runner = Runner(intent_node)
print("intent node up:", intent_node.get_name())


## 2. Drive it from the client side

In [ ]:
require_ros()

from rclpy.event_handler import PublisherEventCallbacks, SubscriptionEventCallbacks

received = []
arrived = threading.Condition()  # the subscription notifies it for every intent that arrives


def on_intent(msg):
    with arrived:
        received.append(json.loads(msg.data))
        arrived.notify_all()


def next_intent(count, timeout_s):
    """The intent that arrives after the first ``count``; None when ``timeout_s`` passes first.

    The waiter is woken by the message itself. Nothing sleeps and looks again.
    """
    with arrived:
        if arrived.wait_for(lambda: len(received) > count, timeout_s):
            return received[count]
        return None


def matched(event):
    """An endpoint's callback that sets ``event`` once it has found a peer."""
    return lambda info: event.set() if info.current_count > 0 else None


# Both ends report when they have found their peer, so nothing waits "for discovery to settle".
pub_matched, sub_matched = threading.Event(), threading.Event()
probe = rclpy.create_node("notebook_probe")
probe.create_subscription(
    String, "/vla/intent", on_intent, 10,
    event_callbacks=SubscriptionEventCallbacks(matched=matched(sub_matched)),
)
transcript_pub = probe.create_publisher(
    String, "/vla/transcript", 10,
    event_callbacks=PublisherEventCallbacks(matched=matched(pub_matched)),
)
probe_runner = Runner(probe)
assert pub_matched.wait(5.0), "nothing subscribes to /vla/transcript"
assert sub_matched.wait(5.0), "nothing publishes on /vla/intent"


def say(text, wait=1.0):
    with arrived:
        n = len(received)
    transcript_pub.publish(String(data=text))
    got = next_intent(n, wait)
    print(f"  {text!r:<38} -> {got}")
    return got


print("transcript -> intent over the wire:")
say("pick up the ball")
say("can you copy my movements")
say("show me your best dance moves")
say("bust a move", wait=3.0)  # free language: only the real brain gets this
say("go home please")

## 3. Prove the two gates work

The counting parser is our witness: a **stop** must produce an intent
*without* a parser call, and **noise** must produce neither.

In [ ]:
require_ros()

calls_before = len(parser_calls)

got = say("stop robot")
assert got and got["command"] == "stop", "stop must be published"
assert len(parser_calls) == calls_before, "stop must NOT invoke the parser!"
after_stop = len(received)
print(
    f"  OK: stop routed in ~0 ms with ZERO parser calls "
    f"(parser count still {len(parser_calls)})"
)

# Classic junk text the noise gate must eat, then a real command. One reliable
# publisher to one node keeps their order, so the next intent is the
# command's when the noise published none.
transcript_pub.publish(String(data="Thank you."))
got = say("wave hello")
assert got and got["command"] == "wave", "noise must publish NO intent"
assert parser_calls[calls_before:] == ["wave hello"], "noise must NOT invoke the parser"
print("  OK: noise dropped - no intent, no parser call")

## 4. Talk to the node, live

The suite above proved the gates. Now drive it yourself: whatever you type
below is published on `/vla/transcript`, crosses the wire, runs the
stop -> noise -> parse gauntlet inside the node, and whatever lands on
`/vla/intent` is printed next to it - along with which path it took (the
parser-call counter is our witness again). Try `pick up the ball`,
`stop robot`, junk like `Thank you.`, and something it has no verb for. Press
**done** to release the cell.


In [ ]:
require_ros()

box = widgets.Text(
    placeholder="say something, e.g. pick up the ball",
    continuous_update=False,  # sync the value once, on Enter/blur
    layout=widgets.Layout(width="420px"),
)
send_btn = widgets.Button(description="send", button_style="primary")
done_btn = widgets.Button(description="done", button_style="danger")
console = widgets.Output(layout=widgets.Layout(border="1px solid #444", padding="6px"))
finished = {"flag": False}
done_btn.on_click(lambda _: finished.update(flag=True))


def send(_=None):
    text = box.value.strip()
    if not text:
        return
    box.value = ""
    with console:
        with arrived:
            n = len(received)
        calls = len(parser_calls)
        transcript_pub.publish(String(data=text))
        got = next_intent(n, 1.5)  # woken by the intent; None when the node published none
        if len(parser_calls) > calls:
            via = "LLM parse"
        elif got and got.get("command") == "stop":
            via = "stop fast path - parser never called"
        else:
            via = "noise gate - dropped, no parser call"
        print(f"you : {text!r}")
        print(f"node: {got}   [{via}]\n")


send_btn.on_click(send)
# Enter (or clicking away) commits the text; the observer sends it.
box.observe(send, "value")
display(widgets.VBox([widgets.HBox([box, send_btn, done_btn]), console]))

# The kernel runs one cell at a time, so this cell hands the buttons' clicks on itself: a
# widget loop on the display's clock, not a wait for a condition.
with ui_events() as poll:
    while not finished["flag"]:
        poll(10)  # keep send/done clicks flowing
        time.sleep(0.05)
print("console closed")

Why so strict about latency? The stop keyword path skips the LLM round
trip *and* every failure mode of a busy server: a queued slot, a slow
generation, a hung connection. Worst case matters more than average case
when the arm is mid-swing.

## 5. The real brain: grammar-constrained parsing (TODO 1.6)

Section 1's node already got the real parser when the server was up.
Under the hood it is one HTTP call to the Gemma 4 llama-server on the iGPU
(`scripts/ensure_voice.sh` brings it up) - with a **GBNF
grammar** attached, so the reply *cannot* be malformed. llama.cpp does *constrained decoding*: at every step, tokens
that would violate the grammar are masked out, and the model physically
cannot emit anything but a list of three strings:

```
["<one of the known commands>", "<object: 0-2 lowercase words>", "<the words heard>"]
```

Parsing failures aren't handled. They're *impossible by construction*.
Three details worth noticing:

- Unknown verbs collapse into `"unknown"` (the grammar *constrains*, the
  system prompt *steers* - `INTENT_SYSTEM_PROMPT` in
  `vla_pipeline/llm/intents.py`).
- The object slot is how *"grab the **bottle**"* carries its noun to the
  behavior that needs it.
- **The order is the latency.** The command and the object come first, the
  words last. The request sets `"stream": True` and the answer is read as
  the server writes it (`answer_pieces`). The moment the text holds the
  command and the object (`call_from_partial`), the robot has its call and
  the reading stops: nothing waits for the words. For a spoken sentence the
  pipeline acts on the call the same way and lets the words follow.

This is TODO 1.6 (`models/llm.py`), and swapping it in for the keyword
stand-in is one constructor argument: the node neither knows nor cares.


In [ ]:
from vla_pipeline.llm.intents import call_from_partial
from vla_pipeline.llm.llama_intent import REQUEST_TIMEOUT_S, answer_pieces

l = cfg["gemma4-e4b-it"]
BASE = f"http://{l['host']}:{l['port']}"
try:
    SERVER_UP = requests.get(f"{BASE}/health", timeout=0.5).status_code == 200
except requests.RequestException:
    SERVER_UP = False

if not SERVER_UP:
    print(f"(no llama-server at {BASE} - the keyword stand-in above keeps the")
    print(" node fully functional. Start the server via ./run_notebooks.sh or")
    print(" scripts/ensure_voice.sh, then re-run.)")
else:
    GRAMMAR = resolve(l["grammar_file"]).read_text()
    http = requests.Session()  # keep-alive: saves ms per command

    def parse_intent(transcript):
        payload = {
            "messages": [
                {"role": "system", "content": INTENT_SYSTEM_PROMPT},
                {"role": "user", "content": transcript.strip()},
            ],
            "temperature": 0.0,  # routing must be deterministic
            "max_tokens": 96,  # the list is short
            "grammar": GRAMMAR,  # <- THE line that makes it robot-grade
            "stream": True,  # <- the answer arrives as it is written
        }
        text = ""
        with http.post(f"{BASE}/v1/chat/completions", json=payload, stream=True, timeout=REQUEST_TIMEOUT_S) as r:
            r.raise_for_status()
            # chunk_size=1: a line is handed on when its last byte arrives
            for piece in answer_pieces(r.iter_lines(chunk_size=1)):
                text += piece
                call = call_from_partial(text)  # None until command and object are complete
                if call is not None:
                    return call, text  # the call is in: stop reading, the words are not needed
        return ParsedCommand(Intent.UNKNOWN), text

    for text in (
        "pick up the ball",
        "can you copy my movements",
        "show me your best dance moves",
        "hold this pencil for me",
        "what's the weather like today",
    ):
        t0 = time.perf_counter()
        call, written = parse_intent(text)
        ms = 1e3 * (time.perf_counter() - t0)
        print(f"  {text!r:<38} -> {call.intent.value} {call.params or ''}   ({ms:.0f} ms, read up to {written!r})")

The time printed is the time to the call, not to the end of the answer:
the command and the object are complete after about a dozen tokens, and the
request is closed there. The last column shows how little of the answer was
read.

## 6. On the rig

The pipeline's node, the class your TODOs subclass:

```bash
# Terminal 1 - real brain (or --mock for the keyword stand-in):
python -m vla_pipeline.llm.intent_node [--mock]

# Terminal 2 - inject language, watch intents:
ros2 topic pub --once /vla/transcript std_msgs/msg/String "{data: pick up the ball}"
ros2 topic echo /vla/intent
```

Anything can publish onto the transcript topic: a human at a terminal, a
test harness, the live UI. Your node neither knows nor cares who is
talking.


In [ ]:
# housekeeping
probe_runner.stop()
node_runner.stop()
if llama_parser is not None:
    llama_parser.close()  # stops a llama-server WE spawned; an externally
    # started one (run_notebooks.sh) keeps running
print("demo nodes stopped")

## 7. Your turn

Open **`project/ros/intent_node.py`**:

| TODO | What | Taught in |
|---|---|---|
| **2.10** | `IntentNode._setup_ros`: String sub/pub, RELIABLE | section 1 |
| **2.11** | `IntentNode._on_transcript`: stop -> noise -> parse -> publish | sections 1 and 3 |

Then run the next cell, which checks your functions, or from a terminal:

```bash
python -m vvla_workshop.selftest_ros

# every component, and the TODOs you have left
python -m vvla_workshop.selftest --quiet
```

The harness publishes transcripts at your node and *counts parser calls*.
A stop that consults the LLM, or noise that slips through, gets named
feedback, not a red X.

That's component 2. Next: `03_integration/01_behavior.ipynb`, where models
and topics become a robot that mirrors you.

In [ ]:
# Your turn: workshop/project/ros/ holds TODOs 2.10-2.11 (intent_node.py). This runs the
# self-test's checks on YOUR classes over ROS 2 - TODO until you implement them.
from vvla_workshop import selftest_ros

selftest_ros.run()
